# 02 — Train the flood model (ConvLSTM)

Reads the cache built by **[01_prepare_data.ipynb](01_prepare_data.ipynb)** and
trains a model that maps **day D−1's GOES imagery** → **day D's flood map** on
the 50 km grid.

**Inputs** (per sample, from the cache):
- GOES sequence `(6, N_CH, 1500, 2500)` — 6 daytime frames × 6 ABI bands, **plus
  the whole-day GLM map as one extra channel when `config.USE_GLM=True`** (off by
  default; `_glm.npy` is always cached, so flipping it needs no rebuild).
- label `y (59, 95)` — scored only on the ~3,360 land cells.

**Model.** A shared per-frame **conv encoder** downsamples each frame to
`IMG // POOL_STRIDE`, a 2-layer **ConvLSTM** fuses the 6 frames, an **exact
pixel→cell pooling** maps the GOES grid onto the 50 km cells, and a small head +
1×1 conv gives the per-cell flood logits.

**Loss/infra.** **Tversky loss** (soft IoU/F1; `α=0.7 > β=0.3`, precision-
favoring to curb over-prediction) on **hard 0/1** targets; **binary metrics**
(F1 / precision / recall / IoU @ threshold) as the val metrics. Lightning
**DDP across both GPUs**, bf16.

> ⚠️ Run top-to-bottom on a **fresh kernel**: `ddp_notebook` forks a worker per
> GPU and needs CUDA uninitialized until `trainer.fit`.

## 1. Config & imports

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# disable NCCL peer-to-peer (this box hangs in P2P init over PCIe) BEFORE any
# CUDA/DDP init. NCCL falls back to shared memory; gradients here are small.
os.environ.setdefault("NCCL_P2P_DISABLE", "1")

# all shared constants live in the repo-root config.py (single source of truth)
ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from config import (BATCH_PER_GPU, CACHE_DIR, CELL_KM, CKPT_DIR, DATA_DIR,
                    EPOCHS, IMG_H, IMG_W, LR, N_BAND, N_CH, POOL_IDX_PATH,
                    POOL_STRIDE, PRED_THRESHOLD, T_FRAMES, TVERSKY_ALPHA,
                    TVERSKY_BETA, USE_GLM, WORKERS, build_grid_cells)

CKPT_DIR.mkdir(parents=True, exist_ok=True)
manifest = pd.read_parquet(CACHE_DIR / "manifest.parquet")
# grid is regenerated from config (not stored) — identical to notebook 01
cells, GRID_R, GRID_C, land_mask = build_grid_cells()
NCELL = GRID_R * GRID_C                  # flat cell count (incl. ocean/empty)
assert POOL_STRIDE in (2, 4), "POOL_STRIDE must divide 1500/2500"
N_POOLS = round(np.log2(POOL_STRIDE))    # encoder max-pools -> feature map at /POOL_STRIDE
print("samples:", manifest["split"].value_counts().reindex(
      ["train", "val", "test"]).to_dict())
print(f"grid {GRID_R}x{GRID_C} = {NCELL} cells ({int(land_mask.sum()):,} land) | "
      f"{CELL_KM} km | input channels {N_CH} (USE_GLM={USE_GLM})")

## 2. Pixel → cell pooling index (one-time, cached)

To register GOES features onto the grid *exactly*, we precompute — for every
location of the encoder's output grid (`IMG // POOL_STRIDE`) — the flat index of
the 50 km cell that geographically contains it (or a **dump bin** for locations
outside CONUS). The model averages features per cell using this index. Built
once and cached.

In [ ]:
import netCDF4
import pyproj
import geopandas as gpd

S_H, S_W = IMG_H // POOL_STRIDE, IMG_W // POOL_STRIDE       # 375 x 625

if POOL_IDX_PATH.exists():
    pool_index = np.load(POOL_IDX_PATH)
    print(f"loaded cached pool index {pool_index.shape} from {POOL_IDX_PATH.name}")
else:
    # ABI fixed-grid geometry from any GOES frame (GOES-16, 2019)
    d0 = manifest["in_day"].min()
    ref = sorted(DATA_DIR.glob(
        f"*/{d0.year}/{d0.month:02d}/{d0.day:02d}/*.nc"))[0]
    
    with netCDF4.Dataset(ref) as nc:
        p = nc["goes_imager_projection"]
        wkt = pyproj.CRS.from_cf({k: p.getncattr(k) for k in p.ncattrs()}).to_wkt()
        h = float(p.perspective_point_height)
        xc = nc["x"][:].astype(np.float64) * h              # 2500 col centres (m)
        yc = nc["y"][:].astype(np.float64) * h              # 1500 row centres (m)

    # block-average ABI centres to the /4 grid, then project geos metres -> lon/lat
    xc4 = xc.reshape(S_W, POOL_STRIDE).mean(1)
    yc4 = yc.reshape(S_H, POOL_STRIDE).mean(1)
    XX, YY = np.meshgrid(xc4, yc4)                           # (375, 625) metres
    to_ll = pyproj.Transformer.from_crs(wkt, "EPSG:4326", always_xy=True)
    lon, lat = to_ll.transform(XX.ravel(), YY.ravel())

    pts = gpd.GeoDataFrame({"pix": np.arange(S_H * S_W)},
                           geometry=gpd.points_from_xy(lon, lat), crs="EPSG:4326")
    j = (gpd.sjoin(pts, cells[["R", "C", "geometry"]], predicate="within", how="left")
         .drop_duplicates("pix").sort_values("pix"))

    # 50 km (R, C) are already north-up (notebook 01) -> flat index R*GRID_C + C
    has = j["R"].notna().to_numpy()
    rr = j["R"].fillna(0).astype(int).to_numpy()
    cc = j["C"].fillna(0).astype(int).to_numpy()
    flat = rr * GRID_C + cc
    pool_index = np.full(S_H * S_W, NCELL, dtype=np.int64)   # NCELL = dump bin
    pool_index[j["pix"].to_numpy()[has]] = flat[has]
    pool_index = pool_index.reshape(S_H, S_W)
    np.save(POOL_IDX_PATH, pool_index)
    print(f"built pool index {pool_index.shape} -> {POOL_IDX_PATH}")

# sanity: how many pixels land in each cell, and are any LAND cells starved?
counts = np.bincount(pool_index.ravel(), minlength=NCELL + 1)[:NCELL]
counts2d = counts.reshape(GRID_R, GRID_C)
starved = int(((counts2d == 0) & land_mask).sum())
print(f"pixels/cell over land: median {np.median(counts2d[land_mask]):.0f}, "
      f"min {counts2d[land_mask].min()}, max {counts2d[land_mask].max()}")
print(f"land cells with NO pixel: {starved}  (these can't be predicted)")

In [ ]:
import matplotlib.pyplot as plt

# the per-cell pixel counts should look like CONUS (registration check)
plt.figure(figsize=(6, 4))
plt.imshow(np.where(land_mask, counts2d, np.nan), cmap="viridis")
plt.title("GOES /4 pixels per 50 km cell (should resemble CONUS)")
plt.colorbar(shrink=0.8, label="pixel count"); plt.xticks([]); plt.yticks([])
plt.tight_layout()

## 3. Dataset — memory-mapped cache

Each `__getitem__` reads the day's `(6, N_BAND, H, W)` GOES band sequence and the
`(59, 95)` label. If **`config.USE_GLM`** is set, the whole-day `_glm.npy` map is
appended as one extra channel (broadcast across the 6 frames) →
`(6, N_CH, H, W)`; otherwise it's bands only. (`_glm.npy` is always cached, so the
toggle needs no rebuild.)

In [ ]:
import time

import torch
from torch.utils.data import DataLoader, Dataset

torch.set_float32_matmul_precision("high")    # TF32 on the Blackwell tensor cores


class FloodCache(Dataset):
    "Reads {date}_x.npy (T, N_BAND, H, W) + {date}_y.npy; appends GLM if USE_GLM."

    def __init__(self, df):
        self.rows = df.reset_index(drop=True)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        d = self.rows["label_day"][i].strftime("%Y%m%d")
        bands = np.load(CACHE_DIR / f"{d}_x.npy")         # (T, N_BAND, H, W) f16
        if USE_GLM:
            glm = np.load(CACHE_DIR / f"{d}_glm.npy")     # (H, W) whole-day map
            x = np.empty((T_FRAMES, N_CH, IMG_H, IMG_W), dtype=np.float16)
            x[:, :N_BAND] = bands
            x[:, N_BAND] = glm                            # broadcast over T frames
        else:
            x = bands
        y = np.load(CACHE_DIR / f"{d}_y.npy").astype(np.float32)
        return torch.from_numpy(x), torch.from_numpy(y)


def split_ds(name):
    return FloodCache(manifest[manifest["split"] == name])


ds_train, ds_val, ds_test = split_ds("train"), split_ds("val"), split_ds("test")

# class balance — informative only (Tversky handles the imbalance, no pos_weight)
tr_y = np.stack([np.load(CACHE_DIR / f"{d:%Y%m%d}_y.npy")
                 for d in manifest.loc[manifest.split == "train", "label_day"]])
POS_RATE = float(tr_y[:, land_mask].mean())

t0 = time.perf_counter()
_x, _y = ds_train[0]
print(f"x {tuple(_x.shape)} {_x.dtype} (USE_GLM={USE_GLM}) | y {tuple(_y.shape)} "
      f"({int(_y.sum())} pos) | {time.perf_counter()-t0:.2f}s")
print(f"train positive rate {POS_RATE:.3%}  "
      f"(Tversky alpha={TVERSKY_ALPHA}, beta={TVERSKY_BETA})")

## 4. Model — conv encoder → ConvLSTM → pixel→cell pool → head

- **Encoder** (shared per frame): 3 conv blocks + max-pools, `N_CH → 24 → 48 →
  64`, down to `IMG // POOL_STRIDE`.
- **ConvLSTM** (2 layers, 64 hidden) fuses the 6 frames.
- **CellPool**: exact geographic scatter-mean of GOES pixels into the 50 km
  cells → `[B, 64, 59, 95]`.
- **Head**: a conv block on the grid + 1×1 → per-cell logits `[B, 59, 95]`.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


def conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False),
        nn.GroupNorm(8, cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False),
        nn.GroupNorm(8, cout), nn.ReLU(inplace=True),
    )


class ConvLSTMCell(nn.Module):
    def __init__(self, c_in, c_hid, k=3):
        super().__init__()
        self.c_hid = c_hid
        self.gates = nn.Conv2d(c_in + c_hid, 4 * c_hid, k, padding=k // 2)

    def forward(self, x, state):
        h, c = state
        i, f, g, o = self.gates(torch.cat([x, h], 1)).chunk(4, 1)
        c = torch.sigmoid(f) * c + torch.sigmoid(i) * torch.tanh(g)
        h = torch.sigmoid(o) * torch.tanh(c)
        return h, c


class ConvLSTM(nn.Module):
    "Stacked ConvLSTM over (B,T,C,H,W); returns the final hidden state."

    def __init__(self, c_in, c_hid, n_layers=2):
        super().__init__()
        self.cells = nn.ModuleList(
            [ConvLSTMCell(c_in if i == 0 else c_hid, c_hid) for i in range(n_layers)])

    def forward(self, x):
        b, t, _, hh, ww = x.shape
        st = [(x.new_zeros(b, cell.c_hid, hh, ww),
               x.new_zeros(b, cell.c_hid, hh, ww)) for cell in self.cells]
        for step in range(t):
            z = x[:, step]
            for li, cell in enumerate(self.cells):
                st[li] = cell(z, st[li])
                z = st[li][0]
        return st[-1][0]


class CellPool(nn.Module):
    "Scatter-mean pixel features into 50 km cells using a precomputed index."

    def __init__(self, pool_index, ncell, grid_r, grid_c):
        super().__init__()
        idx = torch.from_numpy(pool_index.ravel()).long()
        self.ncell, self.gr, self.gc = ncell, grid_r, grid_c
        self.register_buffer("idx", idx)
        self.register_buffer("cnt",
                             torch.bincount(idx, minlength=ncell + 1).float())

    def forward(self, feat):                                # (B,C,H,W)
        b, c, h, w = feat.shape
        flat = feat.reshape(b, c, h * w).float()            # pool in fp32
        out = flat.new_zeros(b, c, self.ncell + 1)
        out.scatter_add_(2, self.idx.view(1, 1, -1).expand(b, c, -1), flat)
        out = out / self.cnt.clamp(min=1).view(1, 1, -1)
        return out[..., :self.ncell].view(b, c, self.gr, self.gc)


class FloodConvLSTM(nn.Module):
    def __init__(self, in_ch=N_CH, enc=(24, 48, 64)):
        super().__init__()
        chans = (in_ch,) + enc
        self.blocks = nn.ModuleList(
            [conv_block(chans[i], chans[i + 1]) for i in range(len(enc))])
        self.lstm = ConvLSTM(enc[-1], enc[-1], n_layers=2)
        self.pool = CellPool(pool_index, NCELL, GRID_R, GRID_C)
        self.head = nn.Sequential(conv_block(enc[-1], 48), nn.Conv2d(48, 1, 1))

    def forward(self, x):                                   # (B,T,C,H,W)
        b, t = x.shape[:2]
        z = x.flatten(0, 1)                                 # (B*T, C, H, W)
        for i, blk in enumerate(self.blocks):
            z = blk(z)
            if i < N_POOLS:                                 # down to /POOL_STRIDE
                z = F.max_pool2d(z, 2)
        z = z.unflatten(0, (b, t))                          # (B, T, 64, h, w)
        h = self.lstm(z)                                    # (B, 64, h, w)
        cells = self.pool(h)                                # (B, 64, GRID_R, GRID_C)
        return self.head(cells)[:, 0]                       # (B, GRID_R, GRID_C)


_n = sum(p.numel() for p in FloodConvLSTM().parameters())
print(f"FloodConvLSTM: {_n:,} parameters")

## 5. LightningModule — Tversky loss, binary metrics

**Tversky** is a soft IoU/F1 — `TI = TP / (TP + α·FP + β·FN)`; we minimize
`1 − TI` over the land cells. It optimizes overlap directly (close to the
F1/IoU we report) and handles the ~2% imbalance without a `pos_weight`.

With **α = 0.7 > β = 0.3** false positives cost more than misses, which
**discourages over-prediction** (flooding big areas). Set `α = β` → Dice, or
`α < β` → favor recall. Targets are **hard 0/1**.

We track **binary metrics at threshold `PRED_THRESHOLD`** — F1, precision,
recall, IoU — plus `val_ap` (threshold-free reference). Checkpoints select on
**best `val_f1`**.

In [ ]:
import lightning as L
from torchmetrics import MetricCollection
from torchmetrics.classification import (BinaryAveragePrecision, BinaryF1Score,
                                         BinaryJaccardIndex, BinaryPrecision,
                                         BinaryRecall)


def soft_tversky(logits, targets, alpha, beta, smooth=1.0):
    "1 - Tversky index over masked cells. alpha=FP weight, beta=FN weight."
    p = torch.sigmoid(logits).float()
    t = targets.float()
    tp = (p * t).sum()
    fp = (p * (1 - t)).sum()
    fn = ((1 - p) * t).sum()
    return 1 - (tp + smooth) / (tp + alpha * fp + beta * fn + smooth)


class FloodNet(L.LightningModule):
    def __init__(self, lr=LR, alpha=TVERSKY_ALPHA, beta=TVERSKY_BETA):
        super().__init__()
        self.save_hyperparameters()
        self.net = FloodConvLSTM()
        self.register_buffer("mask", torch.from_numpy(land_mask))
        # binary metrics at a fixed decision threshold (0/1 predictions)
        self.val_metrics = MetricCollection({
            "val_f1": BinaryF1Score(threshold=PRED_THRESHOLD),
            "val_precision": BinaryPrecision(threshold=PRED_THRESHOLD),
            "val_recall": BinaryRecall(threshold=PRED_THRESHOLD),
            "val_iou": BinaryJaccardIndex(threshold=PRED_THRESHOLD),
        })
        self.val_ap = BinaryAveragePrecision()       # threshold-free reference

    def forward(self, x):
        return self.net(x)

    def _step(self, batch):
        x, y = batch
        logits = self(x)
        loss = soft_tversky(logits[:, self.mask], y[:, self.mask],
                            self.hparams.alpha, self.hparams.beta)
        return loss, logits, y

    def training_step(self, batch, _):
        loss, *_ = self._step(batch)
        self.log("train_loss", loss, prog_bar=True, sync_dist=True)
        return loss

    def validation_step(self, batch, _):
        loss, logits, y = self._step(batch)
        prob = torch.sigmoid(logits[:, self.mask]).flatten()
        tgt = y[:, self.mask].int().flatten()
        self.val_metrics.update(prob, tgt)
        self.val_ap.update(prob, tgt)
        self.log("val_loss", loss, prog_bar=True, sync_dist=True)

    def on_validation_epoch_end(self):
        self.log_dict(self.val_metrics.compute(), prog_bar=True, sync_dist=True)
        self.log("val_ap", self.val_ap.compute(), sync_dist=True)
        self.val_metrics.reset()
        self.val_ap.reset()

    def predict_step(self, batch, _):
        _, logits, y = self._step(batch)
        return (torch.sigmoid(logits[:, self.mask]).float().cpu(),
                y[:, self.mask].cpu())

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.hparams.lr)

## 6. Train — DDP across both GPUs, bf16

In [ ]:
import time

from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger


def make_loader(ds, shuffle, workers=WORKERS):
    return DataLoader(ds, batch_size=BATCH_PER_GPU, shuffle=shuffle,
                      num_workers=workers, prefetch_factor=4,
                      pin_memory=False, persistent_workers=True)


model = FloodNet()
ckpt_cb = ModelCheckpoint(monitor="val_f1", mode="max", save_top_k=1,
                          dirpath=CKPT_DIR, filename="floodnet-{epoch}-{val_f1:.3f}")
trainer = L.Trainer(
    accelerator="gpu", devices=2, strategy="ddp_notebook",
    precision="bf16-mixed", max_epochs=EPOCHS, default_root_dir=str(CKPT_DIR),
    logger=CSVLogger(str(CKPT_DIR), name="logs", flush_logs_every_n_steps=20),
    callbacks=[ckpt_cb], log_every_n_steps=10,
)
t0 = time.perf_counter()
trainer.fit(model, make_loader(ds_train, True), make_loader(ds_val, False, 2))
print(f"fit time {(time.perf_counter()-t0)/60:.1f} min")
print("best checkpoint:", ckpt_cb.best_model_path)

## 7. Training curves

In [ ]:
# locate this run's metrics.csv; fall back to the most recent run under logs/
mpath = Path(trainer.logger.log_dir) / "metrics.csv"
if not mpath.exists():
    runs = sorted(CKPT_DIR.glob("logs/version_*/metrics.csv"))
    mpath = runs[-1] if runs else None

if not mpath or not mpath.exists():
    print("No metrics.csv yet — run section 6 (training) to completion first.\n"
          "CSVLogger flushes it every flush_logs_every_n_steps and at fit end.")
else:
    mdf = pd.read_csv(mpath)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    tr = mdf.dropna(subset=["train_loss"])
    ax[0].plot(tr["step"], tr["train_loss"], lw=0.8, label="train")
    vl = mdf.dropna(subset=["val_loss"])
    ax[0].plot(vl["step"], vl["val_loss"], "o-", label="val")
    ax[0].set_xlabel("step"); ax[0].set_ylabel("Tversky loss (1 - TI)"); ax[0].legend()
    for col, mk in [("val_f1", "o-"), ("val_precision", "s--"),
                    ("val_recall", "^--"), ("val_iou", "d:")]:
        d = mdf.dropna(subset=[col])
        ax[1].plot(d["step"], d[col], mk, label=col.replace("val_", ""))
    ax[1].set_xlabel("step"); ax[1].set_ylabel("binary metric @ 0.5")
    ax[1].set_ylim(0, 1); ax[1].legend()
    plt.tight_layout()
    print("metrics:", mpath)

## 8. Test metrics

Per-cell metrics over every (test day × land cell). Context: the positive rate
itself is a no-skill classifier's precision, and NWS warnings score ≈ 0.24 F1
against observed floods at this granularity.

In [ ]:
from sklearn.metrics import (average_precision_score,
                             classification_report, roc_auc_score)

best = FloodNet.load_from_checkpoint(ckpt_cb.best_model_path)
tester = L.Trainer(accelerator="gpu", devices=1, precision="bf16-mixed",
                   logger=False, enable_checkpointing=False)
out = tester.predict(best, make_loader(ds_test, False))
probs = torch.cat([p for p, _ in out]).numpy().ravel()
y_true = torch.cat([y for _, y in out]).numpy().ravel().astype(int)

print(f"test cells: {len(y_true):,} | positive rate {y_true.mean():.3%}\n")
print(f"PR-AUC (average precision): {average_precision_score(y_true, probs):.4f}")
print(f"ROC-AUC                   : {roc_auc_score(y_true, probs):.4f}\n")
print(classification_report(y_true, probs > 0.5,
                            target_names=["no flood", "flood"], digits=3))

## 9. Look at 10 random test days — prediction vs truth + per-day metrics

For each sampled day: predicted flood probability next to the verified cells,
and that day's own precision / recall / F1 / IoU (at `PRED_THRESHOLD`, over land
cells) so you can see how skill varies day to day.

In [ ]:
test_df = manifest[manifest.split == "test"].reset_index(drop=True)
rng = np.random.default_rng(0)
n_show = min(10, len(ds_test))
idxs = rng.choice(len(ds_test), size=n_show, replace=False)

best_gpu = best.to("cuda:0").eval()
m, thr = land_mask, PRED_THRESHOLD

fig, axes = plt.subplots(n_show, 2, figsize=(12, 3.0 * n_show),
                         constrained_layout=True)
print(f"{'day':>12} {'prec':>6} {'rec':>6} {'F1':>6} {'IoU':>6} "
      f"{'TP':>4} {'FP':>5} {'FN':>4}")
for row, i in enumerate(idxs):
    x, y = ds_test[i]
    day = test_df["label_day"][i].date()
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
        prob = torch.sigmoid(best_gpu(x[None].to("cuda:0")))[0].float().cpu().numpy()
    yt = y.numpy().astype(bool)
    pred = prob >= thr
    tp = int((pred & yt & m).sum())
    fp = int((pred & ~yt & m).sum())
    fn = int((~pred & yt & m).sum())
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0
    iou = tp / (tp + fp + fn) if (tp + fp + fn) else 0.0
    print(f"{str(day):>12} {prec:6.3f} {rec:6.3f} {f1:6.3f} {iou:6.3f} "
          f"{tp:4d} {fp:5d} {fn:4d}")

    im = axes[row, 0].imshow(np.where(m, prob, np.nan), cmap="viridis",
                             vmin=0, vmax=1)
    axes[row, 0].set_title(f"{day}  P(flood)  |  F1 {f1:.2f}  IoU {iou:.2f}",
                           fontsize=9)
    axes[row, 1].imshow(np.where(m, yt, np.nan), cmap="Reds", vmin=0, vmax=1)
    axes[row, 1].set_title(f"truth — {int(yt[m].sum())} flooded cells", fontsize=9)
    for a in axes[row]:
        a.set_xticks([]); a.set_yticks([])

# constrained_layout reserves space for this colorbar, so it never overlaps
# (note: do NOT also call plt.tight_layout() — the two layout engines conflict)
fig.colorbar(im, ax=axes, location="right", shrink=0.35, pad=0.02,
             label="P(flood)")

## Notes & next steps

- **Registration is now exact** — features are pooled into the cell that
  geographically contains each GOES pixel (no bilinear misalignment, no
  vertical flip). The per-cell-count map in §2 is the sanity check.
- **Still flood-days-only** (per the v1 target). Adding quiet (no-flood) days as
  negatives is the most important next lever for real skill.
- **Scale up** once this trains: more years from the cache, deeper encoder,
  longer schedule, and possibly a higher pooling resolution (`POOL_STRIDE=2`).
- Checkpoints + CSV logs under `/mnt/disk1/models/floodnet_convlstm/`.